# Sidewalk Obstacle Footprint Mapping

Takes segmented masks of **sidewalks** and **obstacles** from a street-level image
and produces a **rectified bird's-eye view** showing ground-level obstacle footprints.

**Pipeline:**
1. Load sidewalk + obstacle masks
2. Detect sidewalk edges (excluding image borders)
3. Perspective-rectify to top-down view (horizontal + vertical via vanishing point)
4. Estimate ground-level footprints per obstacle
5. Generate top-view maps

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from skimage.measure import label, regionprops
from PIL import Image
import cv2
import os
import glob
import io
from dotenv import load_dotenv
from google.cloud import storage

## 2. Configuration

- **`IMAGE_NAME`** — blob path of the streetview image in GCS
- **`MASKS_PREFIX`** — GCS prefix for the segmentation results folder (e.g. `segmentation-results/0555_40.971156_29.0509826/forward/`); sidewalk and obstacle masks are auto-discovered from subfolders
- **Footprint parameters** — scan ratios, aspect ratio, max height

In [ ]:
# ── GCS credentials (from .env) ──
load_dotenv(os.path.join("..", ".env"))
GCS_BUCKET_NAME = os.environ.get("GCS_BUCKET_NAME", "")
GCP_PROJECT_ID  = os.environ.get("GCP_PROJECT_ID", "")

# ── Input ──
IMAGE_NAME = "streetview/polygon_4v/20260221T142707Z/0593_forward_40.9720346_29.0545088_123.0.jpg"

filename = os.path.basename(IMAGE_NAME)
img_id, direction, lat, lon, heading = filename.replace(".jpg","").split("_")

MASKS_PREFIX = f"segmentation-results/{img_id}_{lat}_{lon}/{direction}"

# ── Footprint estimation parameters ──
OBSTACLE_IS_TREE = {"tree"}            # types that use trunk-detection logic
FOOTPRINT_BASE_SCAN_RATIO = 0.15      # bottom 15% of non-tree obstacles for base width
TREE_TRUNK_SCAN_RATIO = 0.40          # bottom 40% of trees to find trunk
FOOTPRINT_ASPECT_RATIO = 1.0          # footprint height / width (1.0 = square)
FOOTPRINT_MAX_HEIGHT = 25             # max footprint height in pixels

# ── Scale ──
PIXELS_PER_METER = 1.0  # set to real value if calibrated; otherwise measurements are in pixels

## 3. Load Data

In [ ]:
# ── GCS helpers ──
def _gcs_bucket():
    client = storage.Client(project=GCP_PROJECT_ID)
    return client.bucket(GCS_BUCKET_NAME)

def download_image_from_gcs(blob_name: str) -> np.ndarray:
    """Download an image from GCS and return as numpy array."""
    blob = _gcs_bucket().blob(blob_name)
    data = blob.download_as_bytes()
    return np.array(Image.open(io.BytesIO(data)).convert("RGB"))

def download_mask_from_gcs(blob_name: str) -> np.ndarray:
    """Download a mask from GCS and return as boolean array."""
    blob = _gcs_bucket().blob(blob_name)
    data = blob.download_as_bytes()
    mask = np.array(Image.open(io.BytesIO(data)).convert("L"))
    return (mask > 127).astype(bool)

def list_blobs_under(prefix: str) -> list[str]:
    """List all blob names under a GCS prefix."""
    return [b.name for b in _gcs_bucket().list_blobs(prefix=prefix)]

# ── Fetch original image from GCS ──
print(f"Fetching image from GCS: {IMAGE_NAME}")
original_image = download_image_from_gcs(IMAGE_NAME)
print(f"  shape: {original_image.shape}")

# ── Auto-discover masks from GCS ──
# Structure: {MASKS_PREFIX}/{class}/mask_{idx}.png
print(f"\nListing masks under: {MASKS_PREFIX}/")
all_blobs = list_blobs_under(MASKS_PREFIX + "/")
mask_blobs = [b for b in all_blobs if b.endswith(".png")]

sidewalk_masks = {}      # {"sidewalk_000": mask, "sidewalk_001": mask, ...}
all_obstacle_masks = {}  # {"bollard_000": mask, "tree_001": mask, ...}

for blob_name in sorted(mask_blobs):
    parts = blob_name.split("/")
    class_name = parts[-2]    # "bollard", "sidewalk", "tree", etc.
    mask_file  = parts[-1]    # "mask_000.png"
    idx = mask_file.replace("mask_", "").replace(".png", "")

    print(f"  Downloading {class_name}/mask_{idx}.png ...", end=" ")
    mask = download_mask_from_gcs(blob_name)
    print(f"({mask.sum():,} px)")

    key = f"{class_name}_{idx}"
    if class_name == "sidewalk":
        sidewalk_masks[key] = mask
    else:
        all_obstacle_masks[key] = mask

if not sidewalk_masks:
    raise FileNotFoundError(f"No sidewalk masks found under {MASKS_PREFIX}/sidewalk/")

# ── Assign obstacles to sidewalk segments by proximity ──
# Obstacle masks don't share pixels with sidewalk masks (separate classes),
# so we match by horizontal proximity: find which sidewalk's column range
# is closest to each obstacle's horizontal center.
segments = {}  # {sw_key: {"sidewalk_mask": ..., "obstacle_masks": {...}}}

# Precompute each sidewalk's column range
sw_col_ranges = {}
for sw_key, sw_mask in sidewalk_masks.items():
    segments[sw_key] = {
        "sidewalk_mask": sw_mask,
        "obstacle_masks": {},
    }
    sw_cols = np.where(sw_mask.any(axis=0))[0]
    if len(sw_cols) > 0:
        sw_col_ranges[sw_key] = (sw_cols[0], sw_cols[-1])

for obs_key, obs_mask in all_obstacle_masks.items():
    obs_cols = np.where(obs_mask.any(axis=0))[0]
    if len(obs_cols) == 0:
        continue
    obs_center = (obs_cols[0] + obs_cols[-1]) / 2.0

    best_sw = None
    best_dist = float("inf")
    for sw_key, (c_lo, c_hi) in sw_col_ranges.items():
        # Distance = 0 if obstacle center is within sidewalk column range
        if c_lo <= obs_center <= c_hi:
            dist = 0.0
        else:
            dist = min(abs(obs_center - c_lo), abs(obs_center - c_hi))
        if dist < best_dist:
            best_dist = dist
            best_sw = sw_key

    if best_sw is not None:
        segments[best_sw]["obstacle_masks"][obs_key] = obs_mask

print(f"\n{'='*50}")
print(f"Found {len(sidewalk_masks)} sidewalk segment(s)")
for sw_key, seg in segments.items():
    n_obs = len(seg['obstacle_masks'])
    obs_list = ', '.join(seg['obstacle_masks'].keys()) if n_obs else '(none)'
    print(f"  {sw_key}: {seg['sidewalk_mask'].sum():,} px, {n_obs} obstacles → {obs_list}")

## 4. Helper Functions

In [ ]:
BORDER_MARGIN = 3

def find_row_edges(mask: np.ndarray, border_margin: int = BORDER_MARGIN):
    """
    For each row, find left and right sidewalk edges.
    Rows where the sidewalk touches the image border are extrapolated
    from a linear fit of interior (non-border) edges so that geometry
    extends naturally beyond the visible frame.

    Returns (left_edges, right_edges, valid_mask).
    """
    H, W = mask.shape
    left_edges = np.full(H, np.nan)
    right_edges = np.full(H, np.nan)
    valid = np.zeros(H, dtype=bool)

    # Track which rows have sidewalk pixels but touch the border
    border_rows = np.zeros(H, dtype=bool)
    border_left_clipped = np.zeros(H, dtype=bool)   # left edge is at border
    border_right_clipped = np.zeros(H, dtype=bool)   # right edge is at border

    for r in range(H):
        cols = np.where(mask[r])[0]
        if len(cols) == 0:
            continue
        left_col = cols[0]
        right_col = cols[-1]

        left_at_border = left_col < border_margin
        right_at_border = right_col >= W - border_margin

        if not left_at_border and not right_at_border:
            # Fully interior row — use as-is
            left_edges[r] = left_col
            right_edges[r] = right_col
            valid[r] = True
        else:
            # Row where sidewalk is clipped by the image border
            border_rows[r] = True
            border_left_clipped[r] = left_at_border
            border_right_clipped[r] = right_at_border
            # Store the observed (clipped) values temporarily
            left_edges[r] = left_col
            right_edges[r] = right_col

    # Extrapolate border-clipped edges from valid interior rows
    valid_idx = np.where(valid)[0]
    if len(valid_idx) >= 2:
        # Fit lines to interior edges
        a_L, b_L = np.polyfit(valid_idx.astype(float), left_edges[valid_idx], 1)
        a_R, b_R = np.polyfit(valid_idx.astype(float), right_edges[valid_idx], 1)

        for r in np.where(border_rows)[0]:
            if border_left_clipped[r]:
                # Extrapolate left edge from the trend of interior rows
                left_edges[r] = a_L * r + b_L
            # else: left edge is valid, keep observed value

            if border_right_clipped[r]:
                # Extrapolate right edge from the trend of interior rows
                right_edges[r] = a_R * r + b_R
            # else: right edge is valid, keep observed value

            valid[r] = True

    return left_edges, right_edges, valid


def rectify_sidewalk(image_or_mask, left_edges, right_edges, valid_rows,
                     target_width=None, is_mask=False):
    """
    Per-row warp with horizontal AND vertical perspective correction.
    """
    H, W = image_or_mask.shape[:2]

    valid_widths = right_edges[valid_rows] - left_edges[valid_rows]
    if target_width is None:
        target_width = int(np.median(valid_widths))

    left_interp = np.copy(left_edges)
    right_interp = np.copy(right_edges)
    valid_idx = np.where(valid_rows)[0]
    all_rows = np.arange(H)

    if len(valid_idx) >= 2:
        left_interp = np.interp(all_rows, valid_idx, left_edges[valid_idx])
        right_interp = np.interp(all_rows, valid_idx, right_edges[valid_idx])

    # Vertical perspective correction via vanishing point
    row_scale = np.ones(H, dtype=np.float64)

    if len(valid_idx) >= 2:
        first_valid = valid_idx[0]
        last_valid = valid_idx[-1]

        valid_r = valid_idx.astype(float)
        a_L, b_L = np.polyfit(valid_r, left_edges[valid_idx], 1)
        a_R, b_R = np.polyfit(valid_r, right_edges[valid_idx], 1)

        if abs(a_L - a_R) > 1e-6:
            vy = (b_R - b_L) / (a_L - a_R)
        else:
            vy = -1e8

        ref_dist = abs(last_valid - vy)
        MAX_STRETCH = 6.0
        for r in range(first_valid, last_valid + 1):
            dist = abs(r - vy)
            if dist > 0:
                s = (ref_dist / dist) ** 2
            else:
                s = MAX_STRETCH
            row_scale[r] = min(s, MAX_STRETCH)

    cum_real = np.cumsum(row_scale)
    cum_real = cum_real - cum_real[0]
    total_real_height = cum_real[-1]

    out_height = int(np.ceil(total_real_height)) + 1
    out_rows = np.arange(out_height, dtype=np.float32)
    src_row_for_out = np.interp(out_rows, cum_real, np.arange(H, dtype=np.float32))

    padding = int(target_width * 0.3)
    out_width = target_width + 2 * padding

    map_x = np.zeros((out_height, out_width), dtype=np.float32)
    map_y = np.zeros((out_height, out_width), dtype=np.float32)

    for out_r in range(out_height):
        src_r = src_row_for_out[out_r]
        src_r_int = int(src_r)
        src_r_frac = src_r - src_r_int
        if src_r_int + 1 < H:
            L = left_interp[src_r_int] * (1 - src_r_frac) + left_interp[src_r_int + 1] * src_r_frac
            R = right_interp[src_r_int] * (1 - src_r_frac) + right_interp[src_r_int + 1] * src_r_frac
        else:
            L = left_interp[min(src_r_int, H - 1)]
            R = right_interp[min(src_r_int, H - 1)]

        src_width = R - L
        if src_width <= 0:
            map_x[out_r, :] = -1
            map_y[out_r, :] = src_r
            continue

        out_cols = np.arange(out_width, dtype=np.float32)
        scale = src_width / target_width
        src_cols = L + (out_cols - padding) * scale

        map_x[out_r, :] = src_cols
        map_y[out_r, :] = src_r

    map_x = np.clip(map_x, 0, W - 1)
    map_y = np.clip(map_y, 0, H - 1)

    flags = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    inp = image_or_mask.astype(np.uint8) if is_mask else image_or_mask
    warped = cv2.remap(inp, map_x, map_y, interpolation=flags, borderMode=cv2.BORDER_CONSTANT)

    if is_mask:
        warped = warped.astype(bool)

    return warped, target_width, padding


def estimate_width_footprint(mask: np.ndarray, is_tree: bool = False,
                             base_scan_ratio: float = FOOTPRINT_BASE_SCAN_RATIO,
                             trunk_scan_ratio: float = TREE_TRUNK_SCAN_RATIO,
                             aspect_ratio: float = FOOTPRINT_ASPECT_RATIO,
                             max_height: int = FOOTPRINT_MAX_HEIGHT) -> np.ndarray:
    """
    Estimate the ground-level footprint of obstacles based on their width.
    Trees use trunk_scan_ratio, other obstacles use base_scan_ratio.
    """
    labeled, n = label(mask, return_num=True)
    footprint = np.zeros_like(mask)

    for reg in regionprops(labeled):
        min_row, min_col, max_row, max_col = reg.bbox
        height = max_row - min_row
        component = labeled == reg.label

        if height == 0:
            continue

        if is_tree:
            scan_rows = max(1, int(height * trunk_scan_ratio))
        else:
            scan_rows = max(3, int(height * base_scan_ratio))
        scan_rows = min(scan_rows, height)
        scan_start = max_row - scan_rows

        row_widths = []
        row_centers = []
        for r in range(scan_start, max_row):
            cols = np.where(component[r])[0]
            if len(cols) == 0:
                continue
            w = cols[-1] - cols[0] + 1
            row_widths.append(w)
            row_centers.append((cols[0] + cols[-1]) / 2.0)

        if not row_widths:
            continue

        fp_width = max(int(np.median(row_widths)), 1)
        median_center = np.median(row_centers)

        fp_height = max(1, int(fp_width * aspect_ratio))
        fp_height = min(fp_height, height, max_height)
        fp_top = max_row - fp_height
        fp_left = int(median_center - fp_width / 2.0)
        fp_right = fp_left + fp_width

        fp_left = max(0, fp_left)
        fp_right = min(mask.shape[1], fp_right)
        fp_top = max(0, fp_top)

        footprint[fp_top:max_row, fp_left:fp_right] = True

    return footprint


print("Functions defined: find_row_edges, rectify_sidewalk, estimate_width_footprint")

## 5. Run Pipeline Per Sidewalk Segment

For each sidewalk mask segment, independently:
1. Find sidewalk edges → detect vanishing point
2. Perspective-rectify all masks (horizontal + vertical)
3. Estimate ground-level obstacle footprints
4. Generate top-view maps

In [ ]:
from matplotlib.patches import ConnectionPatch

for seg_idx, (sw_key, seg) in enumerate(segments.items()):
    sidewalk_mask = seg["sidewalk_mask"]
    obstacle_masks_full = seg["obstacle_masks"]
    n_obs = len(obstacle_masks_full)

    print(f"\n{'='*60}")
    print(f"SEGMENT {seg_idx+1}/{len(segments)}: {sw_key}  ({n_obs} obstacles)")
    print(f"{'='*60}")

    # ── Color palette ──
    OBSTACLE_COLORS = {}
    _cmap = plt.cm.get_cmap("tab10", max(n_obs, 1))
    for i, obs_type in enumerate(obstacle_masks_full):
        OBSTACLE_COLORS[obs_type] = _cmap(i)[:3]

    # ── Combined masks (pre-rectification) ──
    obstacle_mask = np.zeros_like(sidewalk_mask, dtype=bool)
    for m in obstacle_masks_full.values():
        obstacle_mask |= m
    effective_mask = sidewalk_mask & ~obstacle_mask

    # ══════════════════════════════════════════════════════════
    # A. Input Visualization
    # ══════════════════════════════════════════════════════════
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(original_image)
    axes[0].set_title("Original Image")

    axes[1].imshow(sidewalk_mask, cmap="Blues")
    axes[1].set_title(f"Sidewalk Mask ({sw_key})")

    overlay = np.zeros((*sidewalk_mask.shape, 3), dtype=np.float32)
    overlay[effective_mask] = [0.2, 0.6, 1.0]
    for obs_type in obstacle_masks_full:
        overlay[obstacle_masks_full[obs_type]] = OBSTACLE_COLORS[obs_type]

    handles = [Patch(facecolor=(0.2, 0.6, 1.0), label="Usable sidewalk")]
    for obs_type in obstacle_masks_full:
        handles.append(Patch(facecolor=OBSTACLE_COLORS[obs_type], label=obs_type))

    axes[2].imshow(overlay)
    axes[2].set_title("Obstacle Silhouettes")
    if handles:
        axes[2].legend(handles=handles, loc="upper right", fontsize=8)

    for ax in axes:
        ax.axis("off")
    plt.suptitle(f"Segment: {sw_key}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

    # ══════════════════════════════════════════════════════════
    # B. Edge Detection
    # ══════════════════════════════════════════════════════════
    left_edges, right_edges, valid_rows = find_row_edges(sidewalk_mask)
    n_valid = valid_rows.sum()
    n_total = sidewalk_mask.any(axis=1).sum()

    print(f"Rows with sidewalk: {n_total}")
    print(f"Rows with valid (non-border) edges: {n_valid}")
    print(f"Rows skipped (edge at photo border): {n_total - n_valid}")

    if n_valid == 0:
        print(f"⚠ No valid edges found for {sw_key} — skipping.")
        continue

    widths_raw = right_edges[valid_rows] - left_edges[valid_rows]
    print(f"Raw sidewalk width range: {widths_raw.min():.0f} — {widths_raw.max():.0f} px")

    # Edge visualization
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(original_image, alpha=0.6)
    sw_overlay = np.zeros((*sidewalk_mask.shape, 4))
    sw_overlay[sidewalk_mask] = [0.2, 0.6, 1.0, 0.3]
    ax.imshow(sw_overlay)

    valid_row_indices = np.where(valid_rows)[0]
    ax.plot(left_edges[valid_rows], valid_row_indices, color="lime", linewidth=1.5, label="Left edge")
    ax.plot(right_edges[valid_rows], valid_row_indices, color="yellow", linewidth=1.5, label="Right edge")

    all_sw_rows = np.where(sidewalk_mask.any(axis=1))[0]
    skipped_rows = np.setdiff1d(all_sw_rows, valid_row_indices)
    if len(skipped_rows) > 0:
        for r in skipped_rows:
            cols = np.where(sidewalk_mask[r])[0]
            if len(cols) > 0:
                ax.plot([cols[0], cols[-1]], [r, r], color="red", linewidth=0.5, alpha=0.4)
        ax.plot([], [], color="red", linewidth=2, alpha=0.4, label=f"Skipped ({len(skipped_rows)})")

    ax.set_title(f"Detected Edges — {sw_key}")
    ax.legend(loc="upper right")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    # ══════════════════════════════════════════════════════════
    # C. Rectification + Footprint Estimation
    # ══════════════════════════════════════════════════════════
    sidewalk_mask_rect, target_w, pad = rectify_sidewalk(
        sidewalk_mask, left_edges, right_edges, valid_rows, is_mask=True)

    obstacle_masks_full_rect = {}
    for obs_type in obstacle_masks_full:
        full_rect, _, _ = rectify_sidewalk(
            obstacle_masks_full[obs_type], left_edges, right_edges, valid_rows,
            target_width=target_w, is_mask=True)
        obstacle_masks_full_rect[obs_type] = full_rect

    obstacle_masks_rect = {}
    for obs_type, full_rect in obstacle_masks_full_rect.items():
        is_tree = any(t in obs_type for t in OBSTACLE_IS_TREE)
        fp = estimate_width_footprint(full_rect, is_tree=is_tree)
        obstacle_masks_rect[obs_type] = fp

        method = "trunk-width" if is_tree else "base-width"
        reduction = (1 - fp.sum() / max(full_rect.sum(), 1)) * 100
        print(f"  '{obs_type}' footprint ({method}): {fp.sum():,} px  "
              f"({reduction:.0f}% reduction)")

    obstacle_mask_rect = np.zeros_like(sidewalk_mask_rect, dtype=bool)
    for m in obstacle_masks_rect.values():
        obstacle_mask_rect |= m
    effective_mask_rect = sidewalk_mask_rect & ~obstacle_mask_rect

    original_image_rect, _, _ = rectify_sidewalk(
        original_image, left_edges, right_edges, valid_rows, target_width=target_w)

    print(f"Rectified size: {sidewalk_mask_rect.shape[1]} × {sidewalk_mask_rect.shape[0]} px")

    # ══════════════════════════════════════════════════════════
    # D. Before vs After Comparison
    # ══════════════════════════════════════════════════════════
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))

    # Top: original perspective
    axes[0, 0].imshow(original_image)
    axes[0, 1].imshow(sidewalk_mask, cmap="Blues")
    overlay_pre = np.zeros((*sidewalk_mask.shape, 3), dtype=np.float32)
    overlay_pre[effective_mask] = [0.2, 0.6, 1.0]
    for ot in obstacle_masks_full:
        c = OBSTACLE_COLORS.get(ot, (1.0, 0.2, 0.2))
        overlay_pre[obstacle_masks_full[ot]] = c
    axes[0, 2].imshow(overlay_pre)

    # Bottom: rectified
    axes[1, 0].imshow(original_image_rect)
    axes[1, 1].imshow(sidewalk_mask_rect, cmap="Blues")

    overlay_rect = np.zeros((*sidewalk_mask_rect.shape, 3), dtype=np.float32)
    overlay_rect[effective_mask_rect] = [0.2, 0.6, 1.0]
    for ot in obstacle_masks_full_rect:
        c = OBSTACLE_COLORS.get(ot, (1.0, 0.2, 0.2))
        overlay_rect[obstacle_masks_full_rect[ot]] = [c[0]*0.4, c[1]*0.4, c[2]*0.4]
    for ot in obstacle_masks_rect:
        c = OBSTACLE_COLORS.get(ot, (1.0, 0.2, 0.2))
        overlay_rect[obstacle_masks_rect[ot]] = c
    axes[1, 2].imshow(overlay_rect)

    left_col = pad
    right_col = pad + target_w
    for ax_idx in range(3):
        axes[1, ax_idx].axvline(x=left_col, color="lime", linewidth=2, linestyle="--", alpha=0.7)
        axes[1, ax_idx].axvline(x=right_col, color="yellow", linewidth=2, linestyle="--", alpha=0.7)

    titles_top = ["Original Image", "Sidewalk Mask", "Silhouettes"]
    titles_bot = ["Rectified Image", "Rectified Sidewalk", "Footprints (rectified)"]
    for i in range(3):
        axes[0, i].set_title(titles_top[i]); axes[0, i].axis("off")
        axes[1, i].set_title(titles_bot[i]); axes[1, i].axis("off")

    plt.suptitle(f"Before vs After Rectification — {sw_key}", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()

    # ══════════════════════════════════════════════════════════
    # E. Top-View Maps
    # ══════════════════════════════════════════════════════════
    sw = sidewalk_mask_rect
    eff = effective_mask_rect

    fig, axes = plt.subplots(1, 3, figsize=(20, 10))

    # Panel 1: footprints
    tv = np.ones((*sw.shape, 3), dtype=np.float32) * 0.15
    tv[sw] = [0.85, 0.85, 0.85]
    tv[eff] = [0.75, 0.9, 1.0]
    for ot, fp in obstacle_masks_rect.items():
        tv[fp] = OBSTACLE_COLORS.get(ot, (1.0, 0.2, 0.2))
    axes[0].imshow(tv)
    axes[0].axvline(x=pad, color="white", linewidth=1.5, alpha=0.6)
    axes[0].axvline(x=pad + target_w, color="white", linewidth=1.5, alpha=0.6)
    axes[0].set_title("Ground Footprints", fontsize=13, fontweight="bold")
    axes[0].axis("off")

    legend_handles = [
        Patch(facecolor=(0.75, 0.9, 1.0), edgecolor="gray", label="Usable sidewalk"),
        Patch(facecolor=(0.85, 0.85, 0.85), edgecolor="gray", label="Sidewalk (blocked)"),
    ]
    for ot in obstacle_masks_rect:
        is_tree = any(t in ot for t in OBSTACLE_IS_TREE)
        method = "trunk" if is_tree else "base"
        legend_handles.append(Patch(facecolor=OBSTACLE_COLORS[ot], label=f"{ot} ({method})"))
    axes[0].legend(handles=legend_handles, loc="upper right", fontsize=8,
                   framealpha=0.9, facecolor="white")

    # Panel 2: full silhouettes
    tv2 = np.ones((*sw.shape, 3), dtype=np.float32) * 0.15
    tv2[sw] = [0.85, 0.85, 0.85]
    for ot, full in obstacle_masks_full_rect.items():
        c = OBSTACLE_COLORS.get(ot, (1.0, 0.2, 0.2))
        tv2[full] = [c[0]*0.6, c[1]*0.6, c[2]*0.6]
    axes[1].imshow(tv2)
    axes[1].axvline(x=pad, color="white", linewidth=1.5, alpha=0.6)
    axes[1].axvline(x=pad + target_w, color="white", linewidth=1.5, alpha=0.6)
    axes[1].set_title("Full Silhouettes (reference)", fontsize=13)
    axes[1].axis("off")

    # Panel 3: overlay
    tv3 = np.ones((*sw.shape, 3), dtype=np.float32) * 0.15
    tv3[sw] = [0.85, 0.85, 0.85]
    tv3[eff] = [0.75, 0.9, 1.0]
    for ot in obstacle_masks_rect:
        c = OBSTACLE_COLORS.get(ot, (1.0, 0.2, 0.2))
        if ot in obstacle_masks_full_rect:
            tv3[obstacle_masks_full_rect[ot]] = [c[0]*0.3, c[1]*0.3, c[2]*0.3]
        tv3[obstacle_masks_rect[ot]] = c
    axes[2].imshow(tv3)
    axes[2].axvline(x=pad, color="white", linewidth=1.5, alpha=0.6)
    axes[2].axvline(x=pad + target_w, color="white", linewidth=1.5, alpha=0.6)
    axes[2].set_title("Footprint + Silhouette Overlay", fontsize=13)
    axes[2].axis("off")

    plt.suptitle(f"Top View — {sw_key}", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()

    # ── Footprint-only view ──
    fig, ax = plt.subplots(figsize=(8, 12))
    tv_fp = np.ones((*sw.shape, 3), dtype=np.float32) * 0.15
    tv_fp[sw] = [0.75, 0.9, 1.0]
    for ot, full in obstacle_masks_full_rect.items():
        fp = obstacle_masks_rect.get(ot, np.zeros_like(full))
        above = full & ~fp
        tv_fp[above] = [0.75, 0.9, 1.0]
    for ot, fp in obstacle_masks_rect.items():
        tv_fp[fp] = OBSTACLE_COLORS.get(ot, (1.0, 0.2, 0.2))

    ax.imshow(tv_fp)
    ax.axvline(x=pad, color="white", linewidth=1.5, alpha=0.6)
    ax.axvline(x=pad + target_w, color="white", linewidth=1.5, alpha=0.6)

    lh = [Patch(facecolor=(0.75, 0.9, 1.0), edgecolor="gray", label="Sidewalk")]
    for ot in obstacle_masks_rect:
        is_tree = any(t in ot for t in OBSTACLE_IS_TREE)
        method = "trunk" if is_tree else "base"
        lh.append(Patch(facecolor=OBSTACLE_COLORS[ot], label=f"{ot} ({method})"))
    ax.legend(handles=lh, loc="upper right", fontsize=9, framealpha=0.9, facecolor="white")

    ax.set_title(f"Footprint Only — {sw_key}", fontsize=14, fontweight="bold")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    # ══════════════════════════════════════════════════════════
    # F. Silhouette → Footprint Mapping with Width Annotations
    # ══════════════════════════════════════════════════════════
    fig, (ax_sil, ax_fp) = plt.subplots(1, 2, figsize=(18, 12))

    # Left panel: full rectified silhouettes
    tv_left = np.ones((*sw.shape, 3), dtype=np.float32) * 0.15
    tv_left[sw] = [0.85, 0.85, 0.85]
    tv_left[eff] = [0.75, 0.9, 1.0]
    for ot, full in obstacle_masks_full_rect.items():
        tv_left[full] = OBSTACLE_COLORS.get(ot, (1.0, 0.2, 0.2))
    ax_sil.imshow(tv_left)
    ax_sil.axvline(x=pad, color="white", linewidth=1, alpha=0.4)
    ax_sil.axvline(x=pad + target_w, color="white", linewidth=1, alpha=0.4)
    ax_sil.set_title("Rectified Silhouettes", fontsize=13, fontweight="bold")
    ax_sil.axis("off")

    # Right panel: ground footprints
    tv_right = np.ones((*sw.shape, 3), dtype=np.float32) * 0.15
    tv_right[sw] = [0.85, 0.85, 0.85]
    tv_right[eff] = [0.75, 0.9, 1.0]
    for ot, fp in obstacle_masks_rect.items():
        tv_right[fp] = OBSTACLE_COLORS.get(ot, (1.0, 0.2, 0.2))
    ax_fp.imshow(tv_right)
    ax_fp.axvline(x=pad, color="white", linewidth=1, alpha=0.4)
    ax_fp.axvline(x=pad + target_w, color="white", linewidth=1, alpha=0.4)
    ax_fp.set_title("Ground Footprints", fontsize=13, fontweight="bold")
    ax_fp.axis("off")

    # Sidewalk width annotation at a few sample rows
    sw_rows_with_data = np.where(sw.any(axis=1))[0]
    if len(sw_rows_with_data) > 2:
        sample_positions = [0.25, 0.5, 0.75]
        for frac in sample_positions:
            sample_row = sw_rows_with_data[int(len(sw_rows_with_data) * frac)]
            sw_cols_at_row = np.where(sw[sample_row])[0]
            if len(sw_cols_at_row) > 0:
                sw_left = sw_cols_at_row[0]
                sw_right = sw_cols_at_row[-1]
                sw_w = sw_right - sw_left
                unit = "m" if PIXELS_PER_METER != 1.0 else "px"
                sw_val = sw_w / PIXELS_PER_METER
                # Draw bracket on footprint panel
                ax_fp.annotate("", xy=(sw_left, sample_row),
                              xytext=(sw_right, sample_row),
                              arrowprops=dict(arrowstyle="<->", color="white",
                                            lw=1.2, shrinkA=0, shrinkB=0))
                ax_fp.text((sw_left + sw_right) / 2, sample_row - 6,
                          f"SW: {sw_val:.0f}{unit}",
                          fontsize=6, color="white", ha="center", va="bottom",
                          bbox=dict(boxstyle="round,pad=0.15", facecolor="gray",
                                  alpha=0.7, edgecolor="none"))

    # Draw connections + width labels for each obstacle
    for ot in obstacle_masks_full_rect:
        full_mask = obstacle_masks_full_rect[ot]
        fp_mask = obstacle_masks_rect.get(ot, np.zeros_like(full_mask))
        color = OBSTACLE_COLORS.get(ot, (1.0, 0.2, 0.2))

        if not full_mask.any() or not fp_mask.any():
            continue

        labeled_sil, n_sil = label(full_mask, return_num=True)
        labeled_fp, n_fp = label(fp_mask, return_num=True)
        sil_regions = regionprops(labeled_sil)
        fp_regions = regionprops(labeled_fp)

        for sil_reg in sil_regions:
            sil_cy, sil_cx = sil_reg.centroid
            sil_bbox = sil_reg.bbox  # (min_row, min_col, max_row, max_col)
            sil_w = sil_bbox[3] - sil_bbox[1]
            sil_h = sil_bbox[2] - sil_bbox[0]

            # Label on silhouette panel
            unit = "m" if PIXELS_PER_METER != 1.0 else "px"
            sil_w_val = sil_w / PIXELS_PER_METER
            sil_h_val = sil_h / PIXELS_PER_METER
            ax_sil.text(sil_cx, sil_bbox[0] - 8, ot,
                       fontsize=7, color="white", ha="center", va="bottom",
                       fontweight="bold",
                       bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.85))
            ax_sil.text(sil_cx, sil_bbox[2] + 10,
                       f"{sil_w_val:.0f}×{sil_h_val:.0f}{unit}",
                       fontsize=6, color="white", ha="center", va="top",
                       bbox=dict(boxstyle="round,pad=0.15", facecolor=(0.2, 0.2, 0.2),
                               alpha=0.7, edgecolor="none"))

            # Find closest footprint component
            best_fp_reg = None
            best_dist = float("inf")
            for fp_reg in fp_regions:
                fp_cy, fp_cx = fp_reg.centroid
                d = abs(sil_cx - fp_cx) + abs(sil_cy - fp_cy) * 0.1
                if d < best_dist:
                    best_dist = d
                    best_fp_reg = fp_reg

            if best_fp_reg is None:
                continue

            fp_cy, fp_cx = best_fp_reg.centroid
            fp_bbox = best_fp_reg.bbox
            fp_w = fp_bbox[3] - fp_bbox[1]
            fp_h = fp_bbox[2] - fp_bbox[0]
            fp_w_val = fp_w / PIXELS_PER_METER
            fp_h_val = fp_h / PIXELS_PER_METER

            # Label on footprint panel
            ax_fp.text(fp_cx, fp_bbox[0] - 8, ot,
                      fontsize=7, color="white", ha="center", va="bottom",
                      fontweight="bold",
                      bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.85))

            # Width bracket below footprint
            bracket_y = fp_bbox[2] + 5
            ax_fp.annotate("", xy=(fp_bbox[1], bracket_y),
                          xytext=(fp_bbox[3], bracket_y),
                          arrowprops=dict(arrowstyle="<->", color="white",
                                        lw=1.5, shrinkA=0, shrinkB=0))
            ax_fp.text(fp_cx, bracket_y + 10,
                      f"w={fp_w_val:.0f}{unit}  h={fp_h_val:.0f}{unit}",
                      fontsize=6, color="white", ha="center", va="top",
                      bbox=dict(boxstyle="round,pad=0.15", facecolor=(0.2, 0.2, 0.2),
                              alpha=0.7, edgecolor="none"))

            # Connection line between panels (silhouette centroid → footprint centroid)
            con = ConnectionPatch(
                xyA=(sil_cx, sil_cy), coordsA=ax_sil.transData,
                xyB=(fp_cx, fp_cy), coordsB=ax_fp.transData,
                color=color, linewidth=2, alpha=0.7,
                arrowstyle="-|>", connectionstyle="arc3,rad=0.15",
                mutation_scale=15)
            fig.add_artist(con)

    # Legend
    map_lh = [
        Patch(facecolor=(0.75, 0.9, 1.0), edgecolor="gray", label="Usable sidewalk"),
        Patch(facecolor=(0.85, 0.85, 0.85), edgecolor="gray", label="Sidewalk (blocked)"),
    ]
    for ot in obstacle_masks_rect:
        is_tree = any(t in ot for t in OBSTACLE_IS_TREE)
        method = "trunk" if is_tree else "base"
        map_lh.append(Patch(facecolor=OBSTACLE_COLORS[ot], label=f"{ot} ({method})"))
    ax_fp.legend(handles=map_lh, loc="upper right", fontsize=8,
                framealpha=0.9, facecolor="white")

    plt.suptitle(f"Silhouette → Footprint Mapping — {sw_key}",
                fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()

    print(f"\n✓ Segment {sw_key} complete.")

## 6. Sidewalk Width Estimation (cm)

Uses the pinhole camera model to convert pixel widths to real-world measurements.

**Camera parameters:** FOV = 90°, pitch = 0°, camera height = 2.5 m

For a pixel at row $v$ in an image of height $H$, the real-world horizontal distance corresponding to $\Delta u$ pixels is:

$$\Delta X = \frac{h \cdot \Delta u}{v - H/2}$$

where $h$ is the camera height. The focal length cancels out (square pixels assumed).

In [ ]:
# ── Camera parameters ──
CAMERA_HEIGHT_M = 2.5     # metres
HFOV_DEG        = 90      # horizontal field of view
PITCH_DEG       = 0       # camera pitch (0 = horizontal)

H_img, W_img = original_image.shape[:2]
cy = H_img / 2   # horizon row (pitch = 0 → image centre)

print(f"Image size : {W_img} × {H_img} px")
print(f"Horizon row: {cy:.0f}")
print(f"Camera height: {CAMERA_HEIGHT_M} m,  HFOV: {HFOV_DEG}°,  pitch: {PITCH_DEG}°\n")

for sw_key, seg in segments.items():
    sw_mask = seg["sidewalk_mask"]
    left_e, right_e, valid_r = find_row_edges(sw_mask)
    valid_idx = np.where(valid_r)[0]

    # keep only rows below the horizon (ground plane)
    valid_idx = valid_idx[valid_idx > cy]
    if len(valid_idx) == 0:
        print(f"{sw_key}: no valid rows below horizon – skipped")
        continue

    # ── per-row width in metres ──
    delta_v   = valid_idx - cy                          # pixels below horizon
    width_px  = right_e[valid_idx] - left_e[valid_idx]  # sidewalk span (px)
    width_m   = CAMERA_HEIGHT_M * width_px / delta_v    # pinhole formula
    width_cm  = width_m * 100

    # ── statistics ──
    med  = np.median(width_cm)
    mean = np.mean(width_cm)
    std  = np.std(width_cm)
    mn, mx = width_cm.min(), width_cm.max()

    print(f"{'='*55}")
    print(f"  Segment : {sw_key}")
    print(f"  Rows    : {len(width_cm)}")
    print(f"  Median  : {med:.1f} cm")
    print(f"  Mean    : {mean:.1f} cm  (σ = {std:.1f} cm)")
    print(f"  Range   : {mn:.1f} – {mx:.1f} cm")
    print(f"{'='*55}\n")

    # ── visualisation ──
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # left: width profile
    ax1.plot(valid_idx, width_cm, "b-", lw=0.8, alpha=0.5, label="Per-row width")
    ax1.axhline(med, color="r", ls="--", label=f"Median {med:.1f} cm")
    ax1.fill_between(valid_idx, mean - std, mean + std,
                     color="blue", alpha=0.08, label=f"±1 σ")
    ax1.set_xlabel("Image row")
    ax1.set_ylabel("Sidewalk width (cm)")
    ax1.set_title(f"Width profile — {sw_key}")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # right: overlay on original image
    ax2.imshow(original_image, alpha=0.7)
    norm = plt.Normalize(vmin=max(0, med - 3*std), vmax=med + 3*std)
    cmap = plt.cm.RdYlGn
    for i, v in enumerate(valid_idx):
        ax2.plot([left_e[v], right_e[v]], [v, v],
                 color=cmap(norm(width_cm[i])), lw=0.6, alpha=0.6)

    # annotate at 25 %, 50 %, 75 % of the valid range
    for frac in (0.25, 0.5, 0.75):
        j = int(len(valid_idx) * frac)
        v = valid_idx[j]
        w = width_cm[j]
        ax2.annotate(
            f"{w:.0f} cm",
            xy=((left_e[v] + right_e[v]) / 2, v),
            fontsize=9, color="white", ha="center",
            bbox=dict(boxstyle="round,pad=0.3", fc="black", alpha=0.7))

    ax2.set_title(f"Width overlay — {sw_key}")
    ax2.axis("off")

    plt.suptitle(f"Sidewalk Width Estimation — {sw_key}",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()